### Search Evaluation

#### Setting up search

In [40]:
import pandas as pd
# Load the ground truth data in the notebook and convert it to a list of dictionaries
df_ground_truth = pd.read_csv("ground_truth-new.csv")

In [41]:
df_ground_truth

,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf
...,...,...
390,Why do I get a 401 Client Error when using the...,4b30b918bc
391,What's the easiest way to force-install reques...,4b30b918bc
392,Can I install requests straight from the GitHu...,4b30b918bc
393,"If pip keeps pulling requests v2.28, what exac...",4b30b918bc


In [45]:
# Convert the ground truth DataFrame to a list of dictionaries
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth)

395

In [20]:
# Download all the necessary files from the GitHub repository to the current working directory
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget {PREFIX}/01-agentic-rag/code/ingest.py
!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

--2026-07-13 12:33:43--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     738  --.-KB/s    in 0s      

2026-07-13 12:33:43 (22.4 MB/s) - ‘ingest.py’ saved [738/738]

--2026-07-13 12:33:43--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1

In [ ]:
# Install the minsearch package that is required by the ingest.py script
%pip install minsearch

In [46]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [24]:
len(documents)

113

In [47]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

#### Collecting relevance data

In [48]:
# Start with one ground truth record:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [49]:
# Run search for this question:
doc_id = q["document"]
query =  q["question"]
search_results = text_search(query)

In [28]:
doc_id


'74eb249bbf'

In [50]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '0fab61eca2',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Is it a group project?',
  'answer': 'No, the capstone is an individual project.\n\nYou can collaborate or discuss a larger idea with other students, but each submitted project must stand on its own. A shared system can work only if it is clearly decomposed into independent projects, where each person has a separate qualifying component and a separate repository.\n\nIf the work cannot be evaluated independently for each participant, it does not satisfy the project requirement.'},
 {'id': '610ccb23c0',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'What happens to code s

In [51]:
# First, compare the retrieved document IDs with the correct document ID:
for d in search_results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


In [52]:
# Then turn this comparison into a relevance list. 
# If the retrieved document ID matches the correct document ID, 
# we assign a relevance score of 1, otherwise 0. 
# This will be used to calculate evaluation metrics like precision and recall.
relevance = []

for d in search_results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [53]:
# Here we put this logic into a function:
def compute_relevance_text(q):
    doc_id = q["document"]
    query =  q["question"]
    search_results = text_search(query)

    relevance = []
    for d in search_results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [54]:
# For the first ground truth record, the relevance list is:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

In [55]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

Where can I watch the live sessions for the course, and how do I ask questions during them?


[1, 0, 0, 0, 0]

In [56]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)

Why does WSL2 say the model needs more memory even though my PC has enough RAM?


[1, 0, 0, 0, 0]

In [57]:
q = ground_truth[56]
print(q["question"])
compute_relevance_text(q)

I see a third-party login failure on the course page — is GitHub login broken right now?


[0, 0, 0, 0, 0]

In [58]:
len(ground_truth)

395

In [59]:
# Now do the same thing for all ground truth questions:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [60]:
# Call it for the first 15 ground truth questions:
# Will get relenvance matrix for the ground truth questions. 
# Each entry in relevance_total_text is a relevance list.
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [61]:
# Look at the results:
# Each entry in relevance_total_text is a relevance list. 
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [64]:
# Next, make the relevance functions generic.
def compute_relevance(q, search_function):
    doc_id = q["document"]
    query = q["question"]
    results = search_function(query)

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [65]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [66]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
example = relevance_total
example

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [67]:
len(ground_truth_sample)
len(ground_truth)

395

In [68]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

In [91]:
example  = relevance_total
example

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0,

##### Search Evaluation Metrics

##### Hit Rate

In [92]:
# Hit Rate tells us if we found the right document, but not where it was.
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

257

In [93]:
cnt / len(example)

0.6506329113924051

In [115]:
# Put the same logic into a function:
def hit_rate(relevance_matrix):
    cnt = 0

    for line in relevance_matrix:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance_matrix)

In [114]:
hit_rate(example)

395


0.6506329113924051

##### Mean Reciprocal Rank (MRR)

In [96]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

226.58333333333334

In [98]:
total_score / len(example)

0.5736286919831224

In [111]:
# Put MRR the same logic into a function:
def mrr(relevance_matrix):
    total_score = 0.0

    for line in relevance_matrix:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 
                total_score = total_score / (rank + 1)
                break

    return total_score / len(relevance_matrix)

In [100]:
len(example)

395

In [112]:
mrr(example)

0.025523051597854996

#### Putting it together

In [109]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [116]:
# We can evaluate any search function:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6506329113924051, 'mrr': 0.025523051597854996}

#### Search Parameter Tuning

##### Trying different boosts

In [118]:
def text_search_v2(query):
    boost_dict = {"question": 0.2, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [119]:
evaluate(
    ground_truth,
    text_search_v2
)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6658227848101266, 'mrr': 0.015999852576443065}

In [128]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [130]:
# Evaluate several boost values:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.6911392405063291, 'mrr': 0.022646539597170855}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.6936708860759494, 'mrr': 0.012137394893058989}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.6506329113924051, 'mrr': 0.025523051597854996}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.6379746835443038, 'mrr': 0.024408882704297395}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.6278481012658228, 'mrr': 0.023759502329000176}


In [131]:
# Define a search function with all three boosts:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [132]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [133]:
results

[{'question': 1.0,
  'answer': 1.0,
  'section': 0.1,
  'hit_rate': 0.7164556962025317,
  'mrr': 0.012120348584981882},
 {'question': 1.0,
  'answer': 1.0,
  'section': 0.2,
  'hit_rate': 0.7164556962025317,
  'mrr': 0.012120285963173176},
 {'question': 1.0,
  'answer': 1.0,
  'section': 0.5,
  'hit_rate': 0.6936708860759494,
  'mrr': 0.012137394893058989},
 {'question': 1.0,
  'answer': 2.0,
  'section': 0.1,
  'hit_rate': 0.7493670886075949,
  'mrr': 0.032795689189620156},
 {'question': 1.0,
  'answer': 2.0,
  'section': 0.2,
  'hit_rate': 0.7468354430379747,
  'mrr': 0.032589754410753076},
 {'question': 1.0,
  'answer': 2.0,
  'section': 0.5,
  'hit_rate': 0.7316455696202532,
  'mrr': 0.03269350909818487},
 {'question': 1.0,
  'answer': 4.0,
  'section': 0.1,
  'hit_rate': 0.7544303797468355,
  'mrr': 0.01712364283951474},
 {'question': 1.0,
  'answer': 4.0,
  'section': 0.2,
  'hit_rate': 0.7544303797468355,
  'mrr': 0.017123642585104258},
 {'question': 1.0,
  'answer': 4.0,
  'sec

In [134]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
34,5.0,10.0,0.2,0.754430,0.032800
33,5.0,10.0,0.1,0.749367,0.032799
3,1.0,2.0,0.1,0.749367,0.032796
19,2.0,4.0,0.2,0.749367,0.032796
35,5.0,10.0,0.5,0.749367,0.032796
18,2.0,4.0,0.1,0.754430,0.032796
5,1.0,2.0,0.5,0.731646,0.032694
4,1.0,2.0,0.2,0.746835,0.032590
20,2.0,4.0,0.5,0.746835,0.032588
28,5.0,2.0,0.2,0.668354,0.025557
